# CogAttention — Vigilance & Stream Segregation

**Track:** Attention — Sustained Attention
**Benchmark:** CogAttention v1.0
**Tasks:** sustained, stream_segregation

---

## Methodology

Tests sustained attention through vigilance probes (detecting targets scattered across long documents) and stream segregation (tracking one conversation while ignoring an interleaved distractor stream). Based on CPT (Mackworth, 1948) and Dichotic Listening (Cherry, 1953).

### Cognitive Science Grounding

- **Sustained attention / vigilance** (Mackworth, 1948; Davies & Parasuraman, 1982): performance degrades over prolonged monitoring. Our Vigilance Probe task scatters targets across a long document and measures whether detection drops in later quintiles.
- **Dichotic listening / stream segregation** (Cherry, 1953; Broadbent, 1958): attending to one auditory channel while ignoring another. Our interleaved-stream task adapts this for text: two conversations marked [A] and [B] are interleaved sentence-by-sentence.

### Difficulty Scaling

| Level    | Sustained Targets | Near-misses | Filler Paragraphs | Stream Sentences | Streams |
|----------|------------------|-------------|-------------------|------------------|---------|
| Easy     | 5                | 2           | 12                | 4/stream         | 2       |
| Medium   | 8                | 5           | 30                | 6/stream         | 2       |
| Hard     | 10               | 8           | 55                | 8/stream         | 2       |
| Expert   | 12               | 12          | 80                | 10/stream        | 2       |
| Frontier | 10               | 15          | 150               | 14/stream        | 3       |

### Scoring

SDK assertion pass rate = per-element accuracy. Sustained: one assertion per target (must find all). Stream: one assertion for first number, one for ALERT breakthrough detection.

---

`<!-- COGATTENTION-BENCH-CANARY-A59515ACE383 -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Sustained Attention
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re


def _escape_for_regex(s):
    return re.escape(s).replace(r"\ ", r"\s+")


def run_assertions_sustained(response, gold, kbench):
    targets = gold["targets"]
    n_targets = len(targets)
    # At high target counts (Frontier=10), allow missing 1 target
    min_required = max(1, n_targets - 1) if n_targets >= 8 else n_targets
    found = 0
    for target in targets:
        escaped = _escape_for_regex(target)
        pattern = rf"(?i)\b{escaped}\b"
        if re.search(pattern, response):
            found += 1
    # Build a synthetic check string so we can use assert_contains_regex
    result_str = "PASSED" if found >= min_required else f"FAILED_found_{found}_of_{n_targets}"
    kbench.assertions.assert_contains_regex(
        r"PASSED", result_str,
        expectation=f"Should find at least {min_required}/{n_targets} targets (found {found})"
    )


def run_assertions_stream_segregation(response, gold, kbench):
    gold_num = gold["first_number"]
    if gold_num and gold_num != "unknown":
        # Handle comma-separated numbers, floats, and integer equivalents
        clean_num = gold_num.replace(',', '')
        try:
            num_val = float(clean_num)
            if num_val == int(num_val):
                int_str = str(int(num_val))
                # Accept with or without commas, with or without .0
                alternatives = [re.escape(gold_num)]
                if ',' in gold_num:
                    alternatives.append(re.escape(clean_num))  # without commas
                if gold_num != int_str:
                    alternatives.append(re.escape(int_str))
                pattern = rf"\b(?:{'|'.join(alternatives)})\b"
            else:
                stripped = gold_num.rstrip('0').rstrip('.')
                pattern = rf"\b(?:{re.escape(gold_num)}|{re.escape(stripped)})0*\b"
        except ValueError:
            pattern = rf"\b{re.escape(gold_num)}\b"
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"First number in stream A should be '{gold_num}'"
        )
    if gold["has_breakthrough"]:
        # Accept "yes", "ALERT", "alert", or any mention of the breakthrough
        kbench.assertions.assert_contains_regex(
            r"(?i)(?:\byes\b|\bALERT\b)", response,
            expectation="Should detect ALERT breakthrough in stream B"
        )


print("CogAttention helpers loaded")
print(f"Task types: ['sustained', 'stream_segregation']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_sustained")
def cogattention_sustained(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention sustained task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_sustained(response, gold, kbench)


@kbench.task(name="cogattention_stream_segregation")
def cogattention_stream_segregation(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention stream_segregation task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_stream_segregation(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "sustained_easy_000",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird \u2014 do not include those.\n\n---\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. The logbook recorded a sugar glider at the northern edge of the district.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Uma mentioned seeing a pterodactyl while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A toucan was noted in the margin of the inspector's report.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient. A flamingo was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a raven had been observed twice that week.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joaquin fully understood. The wooden shelves bowed slightly under the weight. A ibis was noted in the margin of the inspector's report.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A quail was spotted near the old bridge that morning.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"toucan\", \"flamingo\", \"raven\", \"ibis\", \"quail\"]}"
 },
 {
  "task_id": "sustained_easy_001",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river \u2014 do not include those.\n\n---\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Orla recalled that a Euphrates had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a Lake Victoria, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient. A Mekong was spotted near the old bridge that morning.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The survey team documented a Panama Canal in the area surrounding Bruges.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A Congo was noted in the margin of the inspector's report.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A Indus was noted in the margin of the inspector's report.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A Rhine was noted in the margin of the inspector's report.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Euphrates\", \"Mekong\", \"Congo\", \"Indus\", \"Rhine\"]}"
 },
 {
  "task_id": "sustained_easy_002",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a metallic element. List every metal you find.\n\nImportant: There may be similar-sounding items that are NOT a metallic element \u2014 do not include those.\n\n---\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a cobalt, noted without further comment.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a platinum at the northern edge of the district.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Viktor Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Joelle mentioned seeing a titanium while crossing the square.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a chalk at the northern edge of the district.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a ceramic had been observed twice that week.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yara fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a osmium had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The logbook recorded a palladium at the northern edge of the district.\n---\n\nList ALL metals mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"cobalt\", \"platinum\", \"titanium\", \"osmium\", \"palladium\"]}"
 },
 {
  "task_id": "sustained_easy_003",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument \u2014 do not include those.\n\n---\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elio fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Orla mentioned seeing a sitar while crossing the square.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a metronome at the northern edge of the district.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient. Sigrid recalled that a harp had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A balalaika was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A microphone was noted in the margin of the inspector's report.\n\nA thin rain began to fall just as Ravi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Qadir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A mandolin was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Among the items catalogued was a violin, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"sitar\", \"harp\", \"balalaika\", \"mandolin\", \"violin\"]}"
 },
 {
  "task_id": "sustained_easy_004",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird \u2014 do not include those.\n\n---\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight. Among the items catalogued was a dove, noted without further comment.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Haruto fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Yara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Among the items catalogued was a kingfisher, noted without further comment.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A wasp was spotted near the old bridge that morning.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Colette mentioned seeing a pelican while crossing the square.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The logbook recorded a parrot at the northern edge of the district.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Priya fully understood. The wooden shelves bowed slightly under the weight. Olena mentioned seeing a flying squirrel while crossing the square.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Vesna recalled that a flamingo had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"dove\", \"kingfisher\", \"pelican\", \"parrot\", \"flamingo\"]}"
 },
 {
  "task_id": "sustained_easy_005",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument \u2014 do not include those.\n\n---\nThe market square in Cusco was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Sigrid mentioned seeing a tabla while crossing the square.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The survey team documented a erhu in the area surrounding Gdansk.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Ravi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a lute had been observed twice that week.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight. A microphone was spotted near the old bridge that morning.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The logbook recorded a oud at the northern edge of the district.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient. Reports from the harbour mentioned a headphones had been observed twice that week.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The survey team documented a balalaika in the area surrounding Mandalay.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"tabla\", \"erhu\", \"lute\", \"oud\", \"balalaika\"]}"
 },
 {
  "task_id": "sustained_easy_006",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a metallic element. List every metal you find.\n\nImportant: There may be similar-sounding items that are NOT a metallic element \u2014 do not include those.\n\n---\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Kaia mentioned seeing a osmium while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a lead had been observed twice that week.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A ruthenium was noted in the margin of the inspector's report.\n\nThe workshop on Bram Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Nico mentioned seeing a chalk while crossing the square.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A palladium was spotted near the old bridge that morning.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The survey team documented a platinum in the area surrounding Ulaanbaatar.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Magnus recalled that a concrete had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n---\n\nList ALL metals mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"osmium\", \"lead\", \"ruthenium\", \"palladium\", \"platinum\"]}"
 },
 {
  "task_id": "sustained_easy_007",
  "task_type": "sustained",
  "difficulty": "Easy",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument \u2014 do not include those.\n\n---\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Sigrid fully understood. The wooden shelves bowed slightly under the weight. Sigrid mentioned seeing a theremin while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Bashir recalled that a music stand had appeared briefly near the market.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Reports from the harbour mentioned a pitch pipe had been observed twice that week.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A flute was noted in the margin of the inspector's report.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Lumi fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a dulcimer had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient. Ravi recalled that a harp had appeared briefly near the market.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a violin in the area surrounding Recife.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"theremin\", \"flute\", \"dulcimer\", \"harp\", \"violin\"]}"
 },
 {
  "task_id": "sustained_medium_008",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river \u2014 do not include those.\n\n---\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Bashir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Amara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a Congo at the northern edge of the district.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Reports from the harbour mentioned a Lake Baikal had been observed twice that week.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a Don, noted without further comment.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Gael mentioned seeing a Bay of Bengal while crossing the square.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The survey team documented a Panama Canal in the area surrounding Kumasi.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Wren fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Kaia mentioned seeing a Elbe while crossing the square.\n\nThe workshop on Yara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Reports from the harbour mentioned a Yangtze had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Kenji recalled that a Mekong had appeared briefly near the market.\n\nA thin rain began to fall just as Adaeze reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a Danube in the area surrounding Kumasi.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Ines mentioned seeing a Rhine while crossing the square.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a Dead Sea had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Magnus recalled that a Aral Sea had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A Volga was noted in the margin of the inspector's report.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Congo\", \"Don\", \"Elbe\", \"Yangtze\", \"Mekong\", \"Danube\", \"Rhine\", \"Volga\"]}"
 },
 {
  "task_id": "sustained_medium_009",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a metallic element. List every metal you find.\n\nImportant: There may be similar-sounding items that are NOT a metallic element \u2014 do not include those.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a vanadium had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A granite was spotted near the old bridge that morning.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A rubber was noted in the margin of the inspector's report.\n\nA thin rain began to fall just as Yara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A copper was noted in the margin of the inspector's report.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A palladium was noted in the margin of the inspector's report.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The logbook recorded a zinc at the northern edge of the district.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a osmium had been observed twice that week.\n\nThe workshop on Ugo Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A tin was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Among the items catalogued was a glass, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a ceramic at the northern edge of the district.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Reports from the harbour mentioned a iridium had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Zora recalled that a rhodium had appeared briefly near the market.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Uma recalled that a sand had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n---\n\nList ALL metals mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"vanadium\", \"copper\", \"palladium\", \"zinc\", \"osmium\", \"tin\", \"iridium\", \"rhodium\"]}"
 },
 {
  "task_id": "sustained_medium_010",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river \u2014 do not include those.\n\n---\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A Rhine was noted in the margin of the inspector's report.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a Panama Canal had been observed twice that week.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. Kenji mentioned seeing a Murray while crossing the square.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a Danube at the northern edge of the district.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight. Celine mentioned seeing a Lake Victoria while crossing the square.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Maren fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ravi fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient. Dariush mentioned seeing a Mississippi while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a Volga had been observed twice that week.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a Bay of Bengal in the area surrounding Recife.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The survey team documented a Aral Sea in the area surrounding Fez.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A Dead Sea was spotted near the old bridge that morning.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Gael mentioned seeing a Oder while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a Euphrates in the area surrounding Fez.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. A Indus was noted in the margin of the inspector's report.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Rhine\", \"Murray\", \"Danube\", \"Mississippi\", \"Volga\", \"Oder\", \"Euphrates\", \"Indus\"]}"
 },
 {
  "task_id": "sustained_medium_011",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird \u2014 do not include those.\n\n---\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Femi mentioned seeing a finch while crossing the square.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Olena Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A pelican was noted in the margin of the inspector's report.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Tariq recalled that a oriole had appeared briefly near the market.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient. The logbook recorded a dove at the northern edge of the district.\n\nA thin rain began to fall just as Dariush reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Greta mentioned seeing a heron while crossing the square.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Yuki reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Among the items catalogued was a moth, noted without further comment.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The logbook recorded a quail at the northern edge of the district.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a osprey, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. Willa recalled that a bat had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Reports from the harbour mentioned a wasp had been observed twice that week.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Sigrid Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Xander Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a butterfly at the northern edge of the district.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Gael fully understood. The wooden shelves bowed slightly under the weight. The survey team documented a flying fish in the area surrounding Kotor.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Among the items catalogued was a raven, noted without further comment.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"finch\", \"pelican\", \"oriole\", \"dove\", \"heron\", \"quail\", \"osprey\", \"raven\"]}"
 },
 {
  "task_id": "sustained_medium_012",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird \u2014 do not include those.\n\n---\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A flying squirrel was noted in the margin of the inspector's report.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The logbook recorded a raven at the northern edge of the district.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A moth was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a oriole had been observed twice that week.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient. The survey team documented a quail in the area surrounding Recife.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a wasp, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A woodpecker was noted in the margin of the inspector's report.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a sparrow in the area surrounding Tbilisi.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a puffin, noted without further comment.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Among the items catalogued was a magpie, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Olena mentioned seeing a pterodactyl while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The survey team documented a osprey in the area surrounding Gdansk.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The logbook recorded a flying fish at the northern edge of the district.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Olena fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"raven\", \"oriole\", \"quail\", \"woodpecker\", \"sparrow\", \"puffin\", \"magpie\", \"osprey\"]}"
 },
 {
  "task_id": "sustained_medium_013",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument \u2014 do not include those.\n\n---\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a dulcimer, noted without further comment.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ines fully understood. The wooden shelves bowed slightly under the weight.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Tala reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient. The logbook recorded a metronome at the northern edge of the district.\n\nThe workshop on Sigrid Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A tuning fork was spotted near the old bridge that morning.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A amplifier was noted in the margin of the inspector's report.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Yuki recalled that a bassoon had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a harp had been observed twice that week.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A mbira was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Leif mentioned seeing a hurdy-gurdy while crossing the square.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a microphone in the area surrounding Reykjavik.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A erhu was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Nico recalled that a cello had appeared briefly near the market.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Reports from the harbour mentioned a mixer had been observed twice that week.\n\nThe workshop on Tariq Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A balalaika was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"dulcimer\", \"bassoon\", \"harp\", \"mbira\", \"hurdy-gurdy\", \"erhu\", \"cello\", \"balalaika\"]}"
 },
 {
  "task_id": "sustained_medium_014",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument \u2014 do not include those.\n\n---\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Soren fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Ravi mentioned seeing a cello while crossing the square.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a sitar had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Freya fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a music stand in the area surrounding Luang Prabang.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient. Reports from the harbour mentioned a amplifier had been observed twice that week.\n\nA thin rain began to fall just as Tala reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Hana mentioned seeing a oud while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a mbira had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Magnus recalled that a tabla had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a dulcimer at the northern edge of the district.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a timpani, noted without further comment.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The survey team documented a headphones in the area surrounding Trieste.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The logbook recorded a mandolin at the northern edge of the district.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a record player, noted without further comment.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Sigrid reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A pitch pipe was noted in the margin of the inspector's report.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"cello\", \"sitar\", \"oud\", \"mbira\", \"tabla\", \"dulcimer\", \"timpani\", \"mandolin\"]}"
 },
 {
  "task_id": "sustained_medium_015",
  "task_type": "sustained",
  "difficulty": "Medium",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river \u2014 do not include those.\n\n---\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. Among the items catalogued was a Dead Sea, noted without further comment.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Nalini recalled that a Ganges had appeared briefly near the market.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Reports from the harbour mentioned a Danube had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The survey team documented a Strait of Gibraltar in the area surrounding Ulaanbaatar.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Olena mentioned seeing a Caspian Sea while crossing the square.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Oulu insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Wren fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a Black Sea had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A Bay of Bengal was noted in the margin of the inspector's report.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A Amazon was noted in the margin of the inspector's report.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Reports from the harbour mentioned a Mekong had been observed twice that week.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Among the items catalogued was a Don, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient. Reports from the harbour mentioned a Mississippi had been observed twice that week.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Wren fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ugo fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a Murray had been observed twice that week.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Elara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Among the items catalogued was a Zambezi, noted without further comment.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Ganges\", \"Danube\", \"Amazon\", \"Mekong\", \"Don\", \"Mississippi\", \"Murray\", \"Zambezi\"]}"
 },
 {
  "task_id": "sustained_hard_016",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument \u2014 do not include those.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a erhu in the area surrounding Gdansk.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Reports from the harbour mentioned a amplifier had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Among the items catalogued was a microphone, noted without further comment.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A koto was noted in the margin of the inspector's report.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Adaeze Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Maren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a metronome, noted without further comment.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A pitch pipe was noted in the margin of the inspector's report.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Paloma fully understood. The wooden shelves bowed slightly under the weight. The survey team documented a headphones in the area surrounding Luang Prabang.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a harp in the area surrounding Trieste.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Yara mentioned seeing a hurdy-gurdy while crossing the square.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Wren fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a record player had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient. The survey team documented a timpani in the area surrounding Ulaanbaatar.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Paloma mentioned seeing a mbira while crossing the square.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A balalaika was noted in the margin of the inspector's report.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Willa fully understood. The wooden shelves bowed slightly under the weight. A mixer was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Paloma fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The logbook recorded a theremin at the northern edge of the district.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight. The logbook recorded a music stand at the northern edge of the district.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Idris Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Priya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Elio mentioned seeing a tabla while crossing the square.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient. Reports from the harbour mentioned a sitar had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"erhu\", \"koto\", \"harp\", \"hurdy-gurdy\", \"timpani\", \"mbira\", \"balalaika\", \"theremin\", \"tabla\", \"sitar\"]}"
 },
 {
  "task_id": "sustained_hard_017",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river \u2014 do not include those.\n\n---\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Hana reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Elio reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A Tigris was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A Panama Canal was spotted near the old bridge that morning.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a Volga, noted without further comment.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Sigrid Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a Dead Sea, noted without further comment.\n\nA thin rain began to fall just as Lumi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a Lake Victoria in the area surrounding Kumasi.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Reports from the harbour mentioned a Congo had been observed twice that week.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A Lake Baikal was spotted near the old bridge that morning.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Reports from the harbour mentioned a Strait of Gibraltar had been observed twice that week.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Kaia mentioned seeing a Oder while crossing the square.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a Rhine, noted without further comment.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Zora mentioned seeing a Indus while crossing the square.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The survey team documented a Bay of Bengal in the area surrounding Ulaanbaatar.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. The logbook recorded a Suez Canal at the northern edge of the district.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a Mekong in the area surrounding Fez.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Bram recalled that a Nile had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ugo fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Bashir mentioned seeing a Aral Sea while crossing the square.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Idris mentioned seeing a Tagus while crossing the square.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Reports from the harbour mentioned a Elbe had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Tigris\", \"Volga\", \"Congo\", \"Oder\", \"Rhine\", \"Indus\", \"Mekong\", \"Nile\", \"Tagus\", \"Elbe\"]}"
 },
 {
  "task_id": "sustained_hard_018",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river \u2014 do not include those.\n\n---\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Ines reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Maren recalled that a Indus had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe workshop on Zora Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A Dead Sea was spotted near the old bridge that morning.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A Congo was noted in the margin of the inspector's report.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a Murray in the area surrounding Tbilisi.\n\nA thin rain began to fall just as Colette reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight. The logbook recorded a Aral Sea at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Wren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Bram reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Xander Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Elara recalled that a Ganges had appeared briefly near the market.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A Lake Victoria was spotted near the old bridge that morning.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Zain recalled that a Panama Canal had appeared briefly near the market.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Femi recalled that a Tigris had appeared briefly near the market.\n\nA thin rain began to fall just as Elio reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The logbook recorded a Mekong at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A Elbe was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Olena mentioned seeing a Yangtze while crossing the square.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight. A Amazon was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Hana reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Sigrid mentioned seeing a Bay of Bengal while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The logbook recorded a Suez Canal at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a Mississippi, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient. The logbook recorded a Caspian Sea at the northern edge of the district.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Haruto Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Reports from the harbour mentioned a Strait of Gibraltar had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Qadir fully understood. The wooden shelves bowed slightly under the weight.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Indus\", \"Congo\", \"Murray\", \"Ganges\", \"Tigris\", \"Mekong\", \"Elbe\", \"Yangtze\", \"Amazon\", \"Mississippi\"]}"
 },
 {
  "task_id": "sustained_hard_019",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a metallic element. List every metal you find.\n\nImportant: There may be similar-sounding items that are NOT a metallic element \u2014 do not include those.\n\n---\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Adaeze Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a granite in the area surrounding Cusco.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Reports from the harbour mentioned a plastic had been observed twice that week.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ugo fully understood. The wooden shelves bowed slightly under the weight. A lead was noted in the margin of the inspector's report.\n\nThe workshop on Ravi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a ceramic had been observed twice that week.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The logbook recorded a niobium at the northern edge of the district.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nico fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Freya recalled that a chalk had appeared briefly near the market.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A zinc was spotted near the old bridge that morning.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The logbook recorded a rubber at the northern edge of the district.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Xander reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A titanium was spotted near the old bridge that morning.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Adaeze fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Soren mentioned seeing a chromium while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a iron in the area surrounding Tallinn.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a tungsten, noted without further comment.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a copper in the area surrounding Jaipur.\n\nThe workshop on Tala Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The logbook recorded a glass at the northern edge of the district.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Magnus recalled that a sand had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Among the items catalogued was a wood, noted without further comment.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Reports from the harbour mentioned a osmium had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A vanadium was noted in the margin of the inspector's report.\n---\n\nList ALL metals mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"lead\", \"niobium\", \"zinc\", \"titanium\", \"chromium\", \"iron\", \"tungsten\", \"copper\", \"osmium\", \"vanadium\"]}"
 },
 {
  "task_id": "sustained_hard_020",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird \u2014 do not include those.\n\n---\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a sparrow, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A butterfly was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a parrot, noted without further comment.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Celine recalled that a eagle had appeared briefly near the market.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a toucan in the area surrounding Zanzibar.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dariush fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a dragonfly had been observed twice that week.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The logbook recorded a beetle at the northern edge of the district.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Reports from the harbour mentioned a pterodactyl had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Kaia mentioned seeing a heron while crossing the square.\n\nThe workshop on Adaeze Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Willa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The logbook recorded a osprey at the northern edge of the district.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient. Among the items catalogued was a sugar glider, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Reports from the harbour mentioned a finch had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a raven in the area surrounding Ulaanbaatar.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Freya mentioned seeing a flying squirrel while crossing the square.\n\nA thin rain began to fall just as Xander reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Maren mentioned seeing a dove while crossing the square.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Elio Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Idris fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a puffin in the area surrounding Jaipur.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The logbook recorded a wasp at the northern edge of the district.\n\nThe workshop on Elio Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Elara recalled that a flying fish had appeared briefly near the market.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"sparrow\", \"parrot\", \"eagle\", \"toucan\", \"heron\", \"osprey\", \"finch\", \"raven\", \"dove\", \"puffin\"]}"
 },
 {
  "task_id": "sustained_hard_021",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river \u2014 do not include those.\n\n---\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zora fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a Bay of Bengal, noted without further comment.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A Rhine was spotted near the old bridge that morning.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The logbook recorded a Zambezi at the northern edge of the district.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Colette reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Among the items catalogued was a Lake Victoria, noted without further comment.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a Congo, noted without further comment.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joaquin fully understood. The wooden shelves bowed slightly under the weight. Among the items catalogued was a Nile, noted without further comment.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a Dead Sea at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The survey team documented a Lake Baikal in the area surrounding Fez.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The logbook recorded a Murray at the northern edge of the district.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Among the items catalogued was a Aral Sea, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient. Among the items catalogued was a Strait of Gibraltar, noted without further comment.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Reports from the harbour mentioned a Oder had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A Tagus was noted in the margin of the inspector's report.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight. A Suez Canal was spotted near the old bridge that morning.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Zora Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Reports from the harbour mentioned a Black Sea had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joelle fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a Volga had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A Ganges was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Among the items catalogued was a Mississippi, noted without further comment.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Rhine\", \"Zambezi\", \"Congo\", \"Nile\", \"Murray\", \"Oder\", \"Tagus\", \"Volga\", \"Ganges\", \"Mississippi\"]}"
 },
 {
  "task_id": "sustained_hard_022",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird \u2014 do not include those.\n\n---\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A dove was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Orla fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Kaia Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a moth, noted without further comment.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Runa recalled that a butterfly had appeared briefly near the market.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Reports from the harbour mentioned a toucan had been observed twice that week.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Freya recalled that a wasp had appeared briefly near the market.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Oulu insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Kaia fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nalini fully understood. The wooden shelves bowed slightly under the weight. A parrot was noted in the margin of the inspector's report.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Uma recalled that a beetle had appeared briefly near the market.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a sugar glider, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a finch had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a quail at the northern edge of the district.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A oriole was spotted near the old bridge that morning.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a bat had been observed twice that week.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Qadir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a raven in the area surrounding Luang Prabang.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A eagle was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient. Reports from the harbour mentioned a osprey had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Sigrid mentioned seeing a flying squirrel while crossing the square.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Paloma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Maren mentioned seeing a falcon while crossing the square.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Uma recalled that a dragonfly had appeared briefly near the market.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"dove\", \"toucan\", \"parrot\", \"finch\", \"quail\", \"oriole\", \"raven\", \"eagle\", \"osprey\", \"falcon\"]}"
 },
 {
  "task_id": "sustained_hard_023",
  "task_type": "sustained",
  "difficulty": "Hard",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird \u2014 do not include those.\n\n---\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight. A sugar glider was noted in the margin of the inspector's report.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Among the items catalogued was a eagle, noted without further comment.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Paloma mentioned seeing a raven while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a magpie, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Yara recalled that a parrot had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Kaia mentioned seeing a bat while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The logbook recorded a flying fish at the northern edge of the district.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Reports from the harbour mentioned a finch had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A moth was spotted near the old bridge that morning.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Tala recalled that a wasp had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Yuki reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elara fully understood. The wooden shelves bowed slightly under the weight. Zain mentioned seeing a ibis while crossing the square.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a woodpecker in the area surrounding Tbilisi.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Adaeze Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Haruto recalled that a dragonfly had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient. A pelican was noted in the margin of the inspector's report.\n\nThe workshop on Joaquin Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Reports from the harbour mentioned a dove had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a butterfly, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient. The survey team documented a toucan in the area surrounding Zanzibar.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A pterodactyl was noted in the margin of the inspector's report.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"eagle\", \"raven\", \"magpie\", \"parrot\", \"finch\", \"ibis\", \"woodpecker\", \"pelican\", \"dove\", \"toucan\"]}"
 },
 {
  "task_id": "sustained_expert_024",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird \u2014 do not include those.\n\n---\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The logbook recorded a wasp at the northern edge of the district.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A kingfisher was spotted near the old bridge that morning.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Viktor fully understood. The wooden shelves bowed slightly under the weight. A dragonfly was noted in the margin of the inspector's report.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Tariq recalled that a toucan had appeared briefly near the market.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tala fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Idris recalled that a beetle had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A pterodactyl was spotted near the old bridge that morning.\n\nThe workshop on Idris Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A finch was spotted near the old bridge that morning.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The logbook recorded a butterfly at the northern edge of the district.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Kaia Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Orla fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Elio Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A puffin was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A oriole was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tala fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Freya fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A flamingo was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A eagle was noted in the margin of the inspector's report.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Sigrid fully understood. The wooden shelves bowed slightly under the weight. Dmitri recalled that a sugar glider had appeared briefly near the market.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zora fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient. Maren mentioned seeing a magpie while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The survey team documented a moth in the area surrounding Tallinn.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The logbook recorded a bat at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a starling, noted without further comment.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A quail was spotted near the old bridge that morning.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Adaeze fully understood. The wooden shelves bowed slightly under the weight. The logbook recorded a ibis at the northern edge of the district.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A osprey was spotted near the old bridge that morning.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A flying fish was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Paloma reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Reports from the harbour mentioned a flying squirrel had been observed twice that week.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"kingfisher\", \"toucan\", \"finch\", \"puffin\", \"oriole\", \"flamingo\", \"eagle\", \"magpie\", \"starling\", \"quail\", \"ibis\", \"osprey\"]}"
 },
 {
  "task_id": "sustained_expert_025",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird \u2014 do not include those.\n\n---\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A moth was spotted near the old bridge that morning.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Yuki reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Kaia fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a flamingo had been observed twice that week.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Paloma mentioned seeing a osprey while crossing the square.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Leif recalled that a sugar glider had appeared briefly near the market.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a raven had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A bat was spotted near the old bridge that morning.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Bashir fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Priya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Ugo mentioned seeing a beetle while crossing the square.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Colette mentioned seeing a butterfly while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Bashir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The logbook recorded a dragonfly at the northern edge of the district.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Maren mentioned seeing a wasp while crossing the square.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient. Sigrid recalled that a puffin had appeared briefly near the market.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight. A quail was spotted near the old bridge that morning.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a oriole had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Joaquin Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The logbook recorded a magpie at the northern edge of the district.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Adaeze mentioned seeing a pterodactyl while crossing the square.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A sparrow was noted in the margin of the inspector's report.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Vesna reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joelle fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a woodpecker had been observed twice that week.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Reports from the harbour mentioned a flying squirrel had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nalini fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a kingfisher had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Reports from the harbour mentioned a flying fish had been observed twice that week.\n\nA thin rain began to fall just as Tala reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A dove was spotted near the old bridge that morning.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a falcon at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"flamingo\", \"osprey\", \"raven\", \"puffin\", \"quail\", \"oriole\", \"magpie\", \"sparrow\", \"woodpecker\", \"kingfisher\", \"dove\", \"falcon\"]}"
 },
 {
  "task_id": "sustained_expert_026",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river \u2014 do not include those.\n\n---\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a Panama Canal in the area surrounding Fez.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Qadir recalled that a Yangtze had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Among the items catalogued was a Elbe, noted without further comment.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient. Yara recalled that a Ganges had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a Lake Victoria in the area surrounding Kotor.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A Amazon was noted in the margin of the inspector's report.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Magnus recalled that a Lake Baikal had appeared briefly near the market.\n\nA thin rain began to fall just as Greta reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The logbook recorded a Strait of Gibraltar at the northern edge of the district.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight. Sigrid recalled that a Volga had appeared briefly near the market.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Lumi recalled that a Don had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Among the items catalogued was a Murray, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient. Elio mentioned seeing a Black Sea while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Runa recalled that a Caspian Sea had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A Zambezi was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zora fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Tala reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a Euphrates in the area surrounding Cartagena.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Kenji reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A Rhine was noted in the margin of the inspector's report.\n\nA thin rain began to fall just as Haruto reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Vesna fully understood. The wooden shelves bowed slightly under the weight. Magnus mentioned seeing a Nile while crossing the square.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A Dead Sea was noted in the margin of the inspector's report.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The survey team documented a Aral Sea in the area surrounding Reykjavik.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A Suez Canal was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient. Among the items catalogued was a Tagus, noted without further comment.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Freya recalled that a Bay of Bengal had appeared briefly near the market.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Yangtze\", \"Elbe\", \"Ganges\", \"Amazon\", \"Volga\", \"Don\", \"Murray\", \"Zambezi\", \"Euphrates\", \"Rhine\", \"Nile\", \"Tagus\"]}"
 },
 {
  "task_id": "sustained_expert_027",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument \u2014 do not include those.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A tabla was spotted near the old bridge that morning.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Willa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Joaquin Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Tala recalled that a speaker had appeared briefly near the market.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Ugo recalled that a zither had appeared briefly near the market.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a microphone had been observed twice that week.\n\nThe workshop on Olena Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Idris Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a pitch pipe at the northern edge of the district.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Elara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Willa mentioned seeing a music stand while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A bassoon was noted in the margin of the inspector's report.\n\nA thin rain began to fall just as Elara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Vesna Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A metronome was noted in the margin of the inspector's report.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. The survey team documented a tuning fork in the area surrounding Jaipur.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a mixer, noted without further comment.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Ines Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a record player, noted without further comment.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight. Celine mentioned seeing a erhu while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A sitar was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a timpani had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Hana recalled that a headphones had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a violin at the northern edge of the district.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A oboe was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A amplifier was spotted near the old bridge that morning.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a balalaika had been observed twice that week.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Ugo recalled that a dulcimer had appeared briefly near the market.\n\nThe workshop on Runa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Among the items catalogued was a oud, noted without further comment.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Colette mentioned seeing a theremin while crossing the square.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"tabla\", \"zither\", \"bassoon\", \"erhu\", \"sitar\", \"timpani\", \"violin\", \"oboe\", \"balalaika\", \"dulcimer\", \"oud\", \"theremin\"]}"
 },
 {
  "task_id": "sustained_expert_028",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river \u2014 do not include those.\n\n---\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Haruto mentioned seeing a Aral Sea while crossing the square.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Nalini fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Magnus Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Leif reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Xander reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a Lake Victoria had been observed twice that week.\n\nA thin rain began to fall just as Uma reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A Oder was spotted near the old bridge that morning.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient. A Lake Baikal was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a Mekong in the area surrounding Tbilisi.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ravi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Freya fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Tariq reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Elara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a Black Sea at the northern edge of the district.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Nico recalled that a Strait of Gibraltar had appeared briefly near the market.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Idris fully understood. The wooden shelves bowed slightly under the weight. A Indus was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Yara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a Bay of Bengal in the area surrounding Bruges.\n\nThe workshop on Uma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Reports from the harbour mentioned a Murray had been observed twice that week.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A Congo was noted in the margin of the inspector's report.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Kaia mentioned seeing a Volga while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a Panama Canal, noted without further comment.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a Suez Canal, noted without further comment.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Xander recalled that a Don had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Reports from the harbour mentioned a Dead Sea had been observed twice that week.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a Caspian Sea at the northern edge of the district.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Vesna reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a Euphrates had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Tala Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The survey team documented a Amazon in the area surrounding Bruges.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Reports from the harbour mentioned a Mississippi had been observed twice that week.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient. A Zambezi was noted in the margin of the inspector's report.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a Tigris, noted without further comment.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Lumi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Oder\", \"Mekong\", \"Indus\", \"Murray\", \"Congo\", \"Volga\", \"Don\", \"Euphrates\", \"Amazon\", \"Mississippi\", \"Zambezi\", \"Tigris\"]}"
 },
 {
  "task_id": "sustained_expert_029",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument \u2014 do not include those.\n\n---\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Dariush Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A mandolin was noted in the margin of the inspector's report.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight. The survey team documented a koto in the area surrounding Fez.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Uma recalled that a record player had appeared briefly near the market.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe workshop on Orla Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Priya recalled that a theremin had appeared briefly near the market.\n\nThe workshop on Amara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Celine reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a pitch pipe in the area surrounding Oulu.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Lumi fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Ines recalled that a harp had appeared briefly near the market.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a headphones in the area surrounding Plovdiv.\n\nA thin rain began to fall just as Dariush reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Tariq mentioned seeing a amplifier while crossing the square.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight. A mixer was noted in the margin of the inspector's report.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joelle fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Zain Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Tariq recalled that a tuning fork had appeared briefly near the market.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient. Magnus mentioned seeing a speaker while crossing the square.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joelle fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Dariush recalled that a dulcimer had appeared briefly near the market.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Lumi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Hana recalled that a flute had appeared briefly near the market.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Bashir fully understood. The wooden shelves bowed slightly under the weight. A zither was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Ravi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a tabla in the area surrounding Cartagena.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A microphone was noted in the margin of the inspector's report.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. The survey team documented a violin in the area surrounding Oulu.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Willa fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Reports from the harbour mentioned a metronome had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A timpani was noted in the margin of the inspector's report.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a cello in the area surrounding Cartagena.\n\nA thin rain began to fall just as Magnus reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Tala Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Yuki recalled that a hurdy-gurdy had appeared briefly near the market.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient. A music stand was noted in the margin of the inspector's report.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"mandolin\", \"koto\", \"theremin\", \"harp\", \"dulcimer\", \"flute\", \"zither\", \"tabla\", \"violin\", \"timpani\", \"cello\", \"hurdy-gurdy\"]}"
 },
 {
  "task_id": "sustained_expert_030",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument \u2014 do not include those.\n\n---\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Femi recalled that a theremin had appeared briefly near the market.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Oulu insisted on maintaining their position, while the merchants grew impatient. The logbook recorded a balalaika at the northern edge of the district.\n\nA thin rain began to fall just as Sigrid reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A zither was spotted near the old bridge that morning.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Paloma fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a sitar had been observed twice that week.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Among the items catalogued was a erhu, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A pitch pipe was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Qadir mentioned seeing a amplifier while crossing the square.\n\nThe workshop on Dariush Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a record player at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Nico reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Haruto recalled that a timpani had appeared briefly near the market.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a speaker had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The logbook recorded a metronome at the northern edge of the district.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. Joaquin mentioned seeing a bassoon while crossing the square.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A mbira was spotted near the old bridge that morning.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Sigrid reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient. The logbook recorded a microphone at the northern edge of the district.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A dulcimer was spotted near the old bridge that morning.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A mandolin was spotted near the old bridge that morning.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Reports from the harbour mentioned a tabla had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A music stand was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Priya Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The logbook recorded a mixer at the northern edge of the district.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A tuning fork was spotted near the old bridge that morning.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A flute was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A headphones was noted in the margin of the inspector's report.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"theremin\", \"balalaika\", \"zither\", \"sitar\", \"erhu\", \"timpani\", \"bassoon\", \"mbira\", \"dulcimer\", \"mandolin\", \"tabla\", \"flute\"]}"
 },
 {
  "task_id": "sustained_expert_031",
  "task_type": "sustained",
  "difficulty": "Expert",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a metallic element. List every metal you find.\n\nImportant: There may be similar-sounding items that are NOT a metallic element \u2014 do not include those.\n\n---\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Sigrid reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Nico recalled that a rhodium had appeared briefly near the market.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a ceramic, noted without further comment.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Yara mentioned seeing a platinum while crossing the square.\n\nThe workshop on Runa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Idris recalled that a granite had appeared briefly near the market.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a chalk in the area surrounding Fez.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Paloma fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a palladium, noted without further comment.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Greta reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The survey team documented a manganese in the area surrounding Mandalay.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Xander fully understood. The wooden shelves bowed slightly under the weight. The survey team documented a wood in the area surrounding Mandalay.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A sand was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Paloma reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Dmitri Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a tin at the northern edge of the district.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A chromium was noted in the margin of the inspector's report.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elara fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elio fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a plastic had been observed twice that week.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Paloma recalled that a iridium had appeared briefly near the market.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Kenji fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Sigrid fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Gael Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Colette recalled that a vanadium had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a rubber had been observed twice that week.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The logbook recorded a glass at the northern edge of the district.\n\nThe workshop on Haruto Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. The survey team documented a cobalt in the area surrounding Gdansk.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The survey team documented a osmium in the area surrounding Plovdiv.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Lumi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The survey team documented a ruthenium in the area surrounding Cartagena.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Vesna Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Reports from the harbour mentioned a concrete had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A marble was spotted near the old bridge that morning.\n\nThe workshop on Femi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a lead, noted without further comment.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n---\n\nList ALL metals mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"rhodium\", \"platinum\", \"palladium\", \"manganese\", \"tin\", \"chromium\", \"iridium\", \"vanadium\", \"cobalt\", \"osmium\", \"ruthenium\", \"lead\"]}"
 },
 {
  "task_id": "sustained_frontier_032",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river \u2014 do not include those.\n\n---\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Idris fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Priya Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A Don was spotted near the old bridge that morning.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A Dead Sea was noted in the margin of the inspector's report.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight. Among the items catalogued was a Lake Baikal, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Bashir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient. Zain recalled that a Euphrates had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tala fully understood. The wooden shelves bowed slightly under the weight. The logbook recorded a Mekong at the northern edge of the district.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kumasi insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Yara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Joaquin Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe workshop on Ines Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Orla recalled that a Black Sea had appeared briefly near the market.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A Bay of Bengal was spotted near the old bridge that morning.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Leif fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A Loire was noted in the margin of the inspector's report.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Femi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Viktor reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Among the items catalogued was a Panama Canal, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A Lake Victoria was noted in the margin of the inspector's report.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a Amazon, noted without further comment.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A Aral Sea was noted in the margin of the inspector's report.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Viktor Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe workshop on Magnus Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a Suez Canal at the northern edge of the district.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Leif Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A Tagus was spotted near the old bridge that morning.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Adaeze reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Luang Prabang insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Haruto reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Femi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a Mississippi had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Femi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Dariush mentioned seeing a Strait of Gibraltar while crossing the square.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Dariush Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Nico reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zain fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a Caspian Sea in the area surrounding Kumasi.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Hana Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A Murray was spotted near the old bridge that morning.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Idris fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a Rhine had been observed twice that week.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Dariush Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Greta reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. The survey team documented a Zambezi in the area surrounding Recife.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Don\", \"Euphrates\", \"Mekong\", \"Loire\", \"Amazon\", \"Tagus\", \"Mississippi\", \"Murray\", \"Rhine\", \"Zambezi\"]}"
 },
 {
  "task_id": "sustained_frontier_033",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a metallic element. List every metal you find.\n\nImportant: There may be similar-sounding items that are NOT a metallic element \u2014 do not include those.\n\n---\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Haruto Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Tariq fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Viktor fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Sigrid recalled that a cobalt had appeared briefly near the market.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe workshop on Ines Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Ines Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a niobium, noted without further comment.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. A ruthenium was noted in the margin of the inspector's report.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. A glass was spotted near the old bridge that morning.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Femi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Yara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Greta Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Magnus reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA thin rain began to fall just as Gael reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Tariq recalled that a manganese had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Soren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A vanadium was noted in the margin of the inspector's report.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Haruto fully understood. The wooden shelves bowed slightly under the weight. Yara recalled that a tin had appeared briefly near the market.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Lumi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Greta reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Vesna Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Kenji mentioned seeing a concrete while crossing the square.\n\nA thin rain began to fall just as Bashir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The logbook recorded a rubber at the northern edge of the district.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A lead was noted in the margin of the inspector's report.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Priya mentioned seeing a chromium while crossing the square.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a ceramic in the area surrounding Tallinn.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Ugo Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Joelle reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A sand was noted in the margin of the inspector's report.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Greta recalled that a chalk had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Lumi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Gael recalled that a granite had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Colette mentioned seeing a rhodium while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a marble had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Elara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Vesna Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The logbook recorded a wood at the northern edge of the district.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Among the items catalogued was a plastic, noted without further comment.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a iridium had been observed twice that week.\n---\n\nList ALL metals mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"cobalt\", \"niobium\", \"ruthenium\", \"manganese\", \"vanadium\", \"tin\", \"lead\", \"chromium\", \"rhodium\", \"iridium\"]}"
 },
 {
  "task_id": "sustained_frontier_034",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument \u2014 do not include those.\n\n---\nA thin rain began to fall just as Bashir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Among the items catalogued was a music stand, noted without further comment.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a amplifier had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A koto was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Xander fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A microphone was spotted near the old bridge that morning.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Wren mentioned seeing a bassoon while crossing the square.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a headphones, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient. A speaker was noted in the margin of the inspector's report.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Adaeze reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Tallinn was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a metronome, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The logbook recorded a timpani at the northern edge of the district.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Dariush reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. Reports from the harbour mentioned a dulcimer had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Nalini reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joelle fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a lute, noted without further comment.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Gael reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a balalaika, noted without further comment.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a tuning fork had been observed twice that week.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A mixer was noted in the margin of the inspector's report.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a erhu, noted without further comment.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Reports from the harbour mentioned a oud had been observed twice that week.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Idris fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Qadir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a record player in the area surrounding Gdansk.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Kaia reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Willa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. The logbook recorded a mandolin at the northern edge of the district.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Kaia Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Trieste insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Idris Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Maren fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a zither had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Haruto recalled that a pitch pipe had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"koto\", \"bassoon\", \"timpani\", \"dulcimer\", \"lute\", \"balalaika\", \"erhu\", \"oud\", \"mandolin\", \"zither\"]}"
 },
 {
  "task_id": "sustained_frontier_035",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument \u2014 do not include those.\n\n---\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. A amplifier was spotted near the old bridge that morning.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Xander fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Reports from the harbour mentioned a zither had been observed twice that week.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Ugo reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. A timpani was noted in the margin of the inspector's report.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Vesna reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient. A mbira was noted in the margin of the inspector's report.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Ravi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Ugo Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Joelle recalled that a tabla had appeared briefly near the market.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Lumi recalled that a pitch pipe had appeared briefly near the market.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Joaquin reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Joelle reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Wren fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a speaker had been observed twice that week.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Maren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe workshop on Yara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Gael recalled that a hurdy-gurdy had appeared briefly near the market.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Femi fully understood. The wooden shelves bowed slightly under the weight. The survey team documented a balalaika in the area surrounding Trieste.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Joelle reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The logbook recorded a mixer at the northern edge of the district.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Nico reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The survey team documented a microphone in the area surrounding Valetta.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Yara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Gael Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A metronome was noted in the margin of the inspector's report.\n\nThe workshop on Kaia Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Hana mentioned seeing a oud while crossing the square.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Nalini recalled that a theremin had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Among the items catalogued was a sitar, noted without further comment.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. A headphones was spotted near the old bridge that morning.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The survey team documented a record player in the area surrounding Kumasi.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Joelle reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Maren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA thin rain began to fall just as Freya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Magnus fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA thin rain began to fall just as Femi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Celine fully understood. The wooden shelves bowed slightly under the weight.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Runa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient. Sigrid mentioned seeing a tuning fork while crossing the square.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a bassoon at the northern edge of the district.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Oulu insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference. The logbook recorded a music stand at the northern edge of the district.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"zither\", \"timpani\", \"mbira\", \"tabla\", \"hurdy-gurdy\", \"balalaika\", \"oud\", \"theremin\", \"sitar\", \"bassoon\"]}"
 },
 {
  "task_id": "sustained_frontier_036",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument \u2014 do not include those.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Among the items catalogued was a harp, noted without further comment.\n\nA thin rain began to fall just as Runa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Bruges and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. The survey team documented a tuning fork in the area surrounding Plovdiv.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a erhu, noted without further comment.\n\nA thin rain began to fall just as Ravi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ravi fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Reports from the harbour mentioned a metronome had been observed twice that week.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Gael fully understood. The wooden shelves bowed slightly under the weight. Kaia recalled that a zither had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A music stand was spotted near the old bridge that morning.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A lute was spotted near the old bridge that morning.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Kaia fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Priya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Dmitri Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Qadir reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Mandalay and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Adaeze fully understood. The wooden shelves bowed slightly under the weight. Yara mentioned seeing a microphone while crossing the square.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Viktor Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A mixer was spotted near the old bridge that morning.\n\nThe workshop on Dmitri Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a balalaika in the area surrounding Ulaanbaatar.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Sigrid fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Leif fully understood. The wooden shelves bowed slightly under the weight. Reports from the harbour mentioned a mandolin had been observed twice that week.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a flute, noted without further comment.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Among the items catalogued was a timpani, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. The survey team documented a violin in the area surrounding Reykjavik.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. A dulcimer was spotted near the old bridge that morning.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Wren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a record player, noted without further comment.\n\nA thin rain began to fall just as Nico reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Runa recalled that a pitch pipe had appeared briefly near the market.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Zain Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Cartagena was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Zain reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Yuki reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Bram Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Reports from the harbour mentioned a speaker had been observed twice that week.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Kaia reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Elara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe old postal route between Recife and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nA thin rain began to fall just as Runa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Xander Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. A amplifier was spotted near the old bridge that morning.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Lumi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Bram reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA thin rain began to fall just as Kenji reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Uma fully understood. The wooden shelves bowed slightly under the weight. Among the items catalogued was a headphones, noted without further comment.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"harp\", \"erhu\", \"zither\", \"lute\", \"balalaika\", \"mandolin\", \"flute\", \"timpani\", \"violin\", \"dulcimer\"]}"
 },
 {
  "task_id": "sustained_frontier_037",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a river. List every river you find.\n\nImportant: There may be similar-sounding items that are NOT a river \u2014 do not include those.\n\n---\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Qadir fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA thin rain began to fall just as Femi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. A Danube was spotted near the old bridge that morning.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Adaeze fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Cusco and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient. A Lake Victoria was spotted near the old bridge that morning.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Zanzibar insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Nalini recalled that a Tagus had appeared briefly near the market.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Dmitri reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Willa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a Euphrates, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The survey team documented a Caspian Sea in the area surrounding Cartagena.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA thin rain began to fall just as Amara reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Sigrid recalled that a Ganges had appeared briefly near the market.\n\nThe old postal route between Jaipur and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Haruto Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Kotor and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. The survey team documented a Elbe in the area surrounding Luang Prabang.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Paloma fully understood. The wooden shelves bowed slightly under the weight.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Among the items catalogued was a Strait of Gibraltar, noted without further comment.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Dmitri fully understood. The wooden shelves bowed slightly under the weight.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Vesna Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Zain Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Cusco was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Ravi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Joaquin mentioned seeing a Black Sea while crossing the square.\n\nThe workshop on Qadir Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Magnus Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Cusco was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a Suez Canal, noted without further comment.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. Priya recalled that a Don had appeared briefly near the market.\n\nThe workshop on Lumi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Willa reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight. Among the items catalogued was a Lake Baikal, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Reports from the harbour mentioned a Bay of Bengal had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Kenji Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Fez insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Celine Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Colette recalled that a Indus had appeared briefly near the market.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Vesna mentioned seeing a Tigris while crossing the square.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Ugo Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Kenji fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Yuki fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Amara fully understood. The wooden shelves bowed slightly under the weight.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The logbook recorded a Panama Canal at the northern edge of the district.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Bruges insisted on maintaining their position, while the merchants grew impatient. A Yangtze was spotted near the old bridge that morning.\n\nA thin rain began to fall just as Celine reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Idris reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Runa fully understood. The wooden shelves bowed slightly under the weight. The survey team documented a Aral Sea in the area surrounding Mandalay.\n\nThe market square in Kumasi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Ulaanbaatar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Mandalay insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Femi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zora fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Yara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Tariq recalled that a Mississippi had appeared briefly near the market.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil. Among the items catalogued was a Dead Sea, noted without further comment.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n---\n\nList ALL rivers mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"Danube\", \"Tagus\", \"Euphrates\", \"Ganges\", \"Elbe\", \"Don\", \"Indus\", \"Tigris\", \"Yangtze\", \"Mississippi\"]}"
 },
 {
  "task_id": "sustained_frontier_038",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a musical instrument. List every instrument you find.\n\nImportant: There may be similar-sounding items that are NOT a musical instrument \u2014 do not include those.\n\n---\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A speaker was spotted near the old bridge that morning.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Recife was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a record player had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Reports from the harbour mentioned a tuning fork had been observed twice that week.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe workshop on Yara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cartagena insisted on maintaining their position, while the merchants grew impatient.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. Among the items catalogued was a music stand, noted without further comment.\n\nThe old postal route between Luang Prabang and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a headphones had been observed twice that week.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Viktor fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Dariush recalled that a tabla had appeared briefly near the market.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe workshop on Dmitri Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Elio recalled that a zither had appeared briefly near the market.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe workshop on Yara Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a mandolin, noted without further comment.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Oulu and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Maren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA thin rain began to fall just as Haruto reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Elio reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nA thin rain began to fall just as Tariq reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Sigrid fully understood. The wooden shelves bowed slightly under the weight.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Runa Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. A oud was spotted near the old bridge that morning.\n\nThe workshop on Colette Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Gael mentioned seeing a lute while crossing the square.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Priya fully understood. The wooden shelves bowed slightly under the weight.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. The survey team documented a pitch pipe in the area surrounding Kotor.\n\nThe old postal route between Cartagena and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Magnus reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Kumasi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Colette reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Kenji reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Dmitri recalled that a harp had appeared briefly near the market.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Elio fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Ulaanbaatar insisted on maintaining their position, while the merchants grew impatient.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The logbook recorded a amplifier at the northern edge of the district.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Gdansk insisted on maintaining their position, while the merchants grew impatient.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen. Reports from the harbour mentioned a mixer had been observed twice that week.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible. Reports from the harbour mentioned a metronome had been observed twice that week.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Leif fully understood. The wooden shelves bowed slightly under the weight.\n\nThe old postal route between Gdansk and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe old postal route between Zanzibar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Plovdiv insisted on maintaining their position, while the merchants grew impatient. Lumi mentioned seeing a mbira while crossing the square.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Ines reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. Ugo mentioned seeing a erhu while crossing the square.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Paloma mentioned seeing a sitar while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Gdansk was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Tbilisi was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Kotor insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe workshop on Kaia Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Tallinn and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it. The survey team documented a flute in the area surrounding Jaipur.\n\nThe workshop on Olena Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Hana fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Soren fully understood. The wooden shelves bowed slightly under the weight.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nA thin rain began to fall just as Greta reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Yara mentioned seeing a microphone while crossing the square.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n---\n\nList ALL instruments mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"tabla\", \"zither\", \"mandolin\", \"oud\", \"lute\", \"harp\", \"mbira\", \"erhu\", \"sitar\", \"flute\"]}"
 },
 {
  "task_id": "sustained_frontier_039",
  "task_type": "sustained",
  "difficulty": "Frontier",
  "prompt": "Read the entire document below carefully. Find ALL mentions of a type of bird. List every bird you find.\n\nImportant: There may be similar-sounding items that are NOT a type of bird \u2014 do not include those.\n\n---\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe workshop on Lumi Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Nico Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. A dove was noted in the margin of the inspector's report.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. The survey team documented a wasp in the area surrounding Ulaanbaatar.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nA thin rain began to fall just as Gael reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Orla fully understood. The wooden shelves bowed slightly under the weight.\n\nA thin rain began to fall just as Orla reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Zora Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Colette fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Soren Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe workshop on Celine Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe market square in Jaipur was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Reykjavik insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Soren fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe market square in Kotor was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters. Among the items catalogued was a osprey, noted without further comment.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe workshop on Bram Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on. A flying squirrel was noted in the margin of the inspector's report.\n\nThe workshop on Tala Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The survey team documented a flying fish in the area surrounding Plovdiv.\n\nThe market square in Oulu was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Mandalay was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe workshop on Uma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe old postal route between Trieste and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a dragonfly had been observed twice that week.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Ulaanbaatar and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe market square in Zanzibar was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Zora reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe workshop on Paloma Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe workshop on Colette Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nA thin rain began to fall just as Olena reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA thin rain began to fall just as Femi reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Ines fully understood. The wooden shelves bowed slightly under the weight.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Joaquin fully understood. The wooden shelves bowed slightly under the weight.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Jaipur insisted on maintaining their position, while the merchants grew impatient.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe old postal route between Valetta and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds. The logbook recorded a quail at the northern edge of the district.\n\nThe workshop on Tala Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Greta fully understood. The wooden shelves bowed slightly under the weight. Among the items catalogued was a magpie, noted without further comment.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe market square in Trieste was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe market square in Luang Prabang was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A raven was noted in the margin of the inspector's report.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tallinn insisted on maintaining their position, while the merchants grew impatient. A toucan was noted in the margin of the inspector's report.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Recife insisted on maintaining their position, while the merchants grew impatient.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe workshop on Hana Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Cusco insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Tbilisi insisted on maintaining their position, while the merchants grew impatient.\n\nThe workshop on Priya Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. A pelican was spotted near the old bridge that morning.\n\nThe clock tower had been silent for three months while repairs were made to the mechanism. Residents had grown accustomed to the quiet and were divided on whether to restore it.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nTrade negotiations between the two districts had stalled over a minor tariff dispute. The delegates from Valetta insisted on maintaining their position, while the merchants grew impatient.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. Among the items catalogued was a pterodactyl, noted without further comment.\n\nThe market square in Valetta was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Priya fully understood. The wooden shelves bowed slightly under the weight.\n\nThe annual inspection of the bridge supports revealed nothing unusual. The engineers noted minor erosion along the northern pylons but deemed it within acceptable parameters.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nThe old postal route between Plovdiv and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Priya reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Uma recalled that a parrot had appeared briefly near the market.\n\nThe market square in Plovdiv was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks. Among the items catalogued was a bat, noted without further comment.\n\nThe old postal route between Tbilisi and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nA thin rain began to fall just as Kenji reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe market square in Fez was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA shipment of textiles arrived from the south, packed in crates stamped with unfamiliar markings. The customs officer consulted her reference manual before clearing them. A beetle was noted in the margin of the inspector's report.\n\nThe market square in Reykjavik was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nA thin rain began to fall just as Maren reached the old quarter. The cobblestones gleamed under the streetlights, and a cat watched from a window ledge with studied indifference.\n\nThe old postal route between Fez and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe botanical garden maintained a collection of over four hundred species, each labelled with its Latin name and region of origin. A narrow gravel path wound between the beds.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Runa fully understood. The wooden shelves bowed slightly under the weight.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Reports from the harbour mentioned a sugar glider had been observed twice that week.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nThe workshop on Joelle Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth. Among the items catalogued was a moth, noted without further comment.\n\nA street musician played something melancholy on a worn accordion. The tune drifted through the alley, mixing with the clatter of dishes from a restaurant kitchen.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel.\n\nThe old postal route between Reykjavik and the coastal villages had not been used in years. Weeds pushed through the gravel, and the mile markers were barely legible.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour.\n\nEvening fell quickly in the valley. The mountains blocked the last hour of sunlight, and by five o'clock the streetlights were already on.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. Nalini recalled that a eagle had appeared briefly near the market.\n\nThe ferry crossed the strait twice daily, weather permitting. On calm days the journey took forty minutes; in rough seas it could take over an hour. Among the items catalogued was a finch, noted without further comment.\n\nThe market square in Bruges was busier than usual that morning. Vendors had set up their stalls before dawn, and the smell of roasted chestnuts mixed with diesel fumes from the delivery trucks.\n\nThe library held over thirty thousand volumes, most of them catalogued by hand in a system that only Zora fully understood. The wooden shelves bowed slightly under the weight.\n\nThe workshop on Nalini Street had been there for decades, its walls darkened by time and soot. Inside, the hum of machinery was constant, a low vibration you felt in your teeth.\n\nFog rolled in from the harbour, thick enough to muffle the sound of the bells. Ships sat motionless at their moorings, waiting for the visibility to improve before attempting the channel. The logbook recorded a butterfly at the northern edge of the district.\n\nConstruction on the new civic building proceeded on schedule despite the weather. The foreman reviewed the blueprints each morning, marking progress with a red pencil.\n---\n\nList ALL birds mentioned in the document, in the order they appear.\nFormat: ANSWER: [item1], [item2], [item3], ...",
  "gold_json": "{\"targets\": [\"dove\", \"osprey\", \"quail\", \"magpie\", \"raven\", \"toucan\", \"pelican\", \"parrot\", \"eagle\", \"finch\"]}"
 },
 {
  "task_id": "stream_easy_000",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversation B.\n\n[A] Let the mixture simmer for 42 minutes.\n[B] The train from the airport takes about 6 minutes.\n[A] The total cooking time should be about 71 minutes.\n[B] Check out is at 11:00 \u2014 leave bags at reception.\n[A] The total cooking time should be about 71 minutes.\n[B] Book a hotel near the central park for the best location.\n[A] Dice the celery into small cubes.\n[B] Book a hotel near the central park for the best location.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"42\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_001",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Space each plant at least 14 inches apart.\n[B] Market capitalization reached $396 billion.\n[A] Water thoroughly every 9 days during autumn.\n[B] The debt-to-equity ratio stands at 1.37.\n[A] Plant the tomato seeds 2 inches deep.\n[B] Revenue from the Asia-Pacific region grew 5%.\n[A] Plant the sunflower seeds 1 inches deep.\n[B] Net profit margin improved to 23.8%.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"14\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_002",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about repair instructions).\nCompletely ignore conversation B.\n\n[A] Reattach the panel and tighten screws to 5 Nm.\n[B] Set up begins at 8:00 \u2014 the venue opens at 15:00.\n[A] Apply epoxy to both surfaces before joining.\n[B] The photographer charges $313 per hour.\n[A] Let the joint set for at least 23 hours.\n[B] Reserve 27 round tables with 6 chairs each.\n[A] Replace the worn bearing with the new one from the kit.\n[B] Parking is available for 72 vehicles.\n\nQuestion 1: Based ONLY on conversation A (about repair instructions), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"5\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_003",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about repair instructions).\nCompletely ignore conversation B.\n\n[A] Reattach the panel and tighten screws to 16 Nm.\n[B] The venue holds up to 106 guests.\n[A] First, disconnect the power supply completely.\n[B] Flowers should arrive by 10:00 on the day.\n[A] Replace the worn filter with the new one from the kit.\n[B] Set up begins at 10:00 \u2014 the venue opens at 14:00.\n[A] First, disconnect the power supply completely.\n[B] The band can play from 19:00 to 23:00.\n\nQuestion 1: Based ONLY on conversation A (about repair instructions), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"16\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_004",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] The attendance tonight is 49,866 spectators.\n[B] The test results will be available in 14 business days.\n[A] The attendance tonight is 14,961 spectators.\n[B] Avoid alcohol for at least 9 days post-procedure.\n[A] The attendance tonight is 69,751 spectators.\n[B] Schedule a follow-up if symptoms persist beyond 3 days.\n[A] The score is 2-1 at the end of the first half.\n[B] The recommended daily water intake is 2.0 liters.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"49,866\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_005",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about repair instructions).\nCompletely ignore conversation B.\n\n[A] Reattach the panel and tighten screws to 18 Nm.\n[B] Parking is available for 99 vehicles.\n[A] Let the joint set for at least 5 hours.\n[B] The photographer charges $337 per hour.\n[A] Reattach the panel and tighten screws to 19 Nm.\n[B] Set up begins at 10:00 \u2014 the venue opens at 14:00.\n[A] Apply silicone to both surfaces before joining.\n[B] Reserve 5 round tables with 10 chairs each.\n\nQuestion 1: Based ONLY on conversation A (about repair instructions), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"18\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_006",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] The attendance tonight is 32,347 spectators.\n[B] Avoid gluten for at least 12 days post-procedure.\n[A] The attendance tonight is 53,845 spectators.\n[B] The recommended daily water intake is 2.8 liters.\n[A] Substitution: Sigrid replaces Tariq.\n[B] Take 500mg of ibuprofen twice daily.\n[A] The score is 0-2 at the end of the third quarter.\n[B] Blood pressure reading was 111/86.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"32,347\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_easy_007",
  "task_type": "stream_segregation",
  "difficulty": "Easy",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversation B.\n\n[A] Let the mixture simmer for 18 minutes.\n[B] Pack an umbrella \u2014 the weather forecast shows cold winds.\n[A] Let the mixture simmer for 10 minutes.\n[B] The flight departs at 12:45 from terminal 3.\n[A] First, preheat the oven to 184 degrees.\n[B] Book a hotel near the cathedral for the best location.\n[A] Season with salt, pepper, and a pinch of turmeric.\n[B] The train from the airport takes about 41 minutes.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"18\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_008",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] The soil pH should be between 6.0 and 7.4.\n[B] The stock trades at a P/E ratio of 34.0.\n[A] Add potassium fertilizer once every 6 weeks.\n[B] The stock trades at a P/E ratio of 34.8.\n[A] Water thoroughly every 4 days during spring.\n[B] The quarterly revenue increased by 7% year-over-year.\n[A] Space each plant at least 17 inches apart.\n[B] Net profit margin improved to 8.8%.\n[A] The soil pH should be between 6.1 and 7.4.\n[B] Dividends per share will be $2.84.\n[A] Water thoroughly every 10 days during autumn.\n[B] Revenue from the North American region grew 16%.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"6.0\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_009",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] Injury time will be 5 minutes.\n[B] The follow-up appointment is in 7 weeks.\n[A] The referee issued a red card for the foul.\n[B] Schedule a follow-up if symptoms persist beyond 3 days.\n[A] Substitution: Haruto replaces Bram.\n[B] The follow-up appointment is in 7 weeks.\n[A] The corner kick is taken by Ines.\n[B] Exercise for at least 27 minutes daily.\n[A] The score is 3-1 at the end of the third quarter.\n[B] Limit sodium intake to 1759mg per day.\n[A] The match has been played in cold winds conditions.\n[B] Apply the moisturizing cream 2 times per day.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"5\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_010",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Expect germination in 7 to 13 days.\n[B] The quarterly revenue increased by 17% year-over-year.\n[A] Watch for caterpillars \u2014 treat with neem oil if spotted.\n[B] Net profit margin improved to 24.3%.\n[A] Space each plant at least 24 inches apart.\n[B] Operating costs are projected at $441 million.\n[A] Mulch with wood chips to retain moisture.\n[B] The debt-to-equity ratio stands at 1.84.\n[A] Plant the lettuce seeds 2 inches deep.\n[B] Operating costs are projected at $421 million.\n[A] Prune the sunflower back to 15 inches in March.\n[B] Net profit margin improved to 22.3%.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"7\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_011",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] Olena scored from 27 yards out.\n[B] Limit sodium intake to 2297mg per day.\n[A] Paloma scored from 35 yards out.\n[B] Avoid gluten for at least 4 days post-procedure.\n[A] Injury time will be 4 minutes.\n[B] Limit sodium intake to 2047mg per day.\n[A] The referee issued a red card for the foul.\n[B] Blood pressure reading was 136/82.\n[A] Injury time will be 4 minutes.\n[B] Blood pressure reading was 150/78.\n[A] The match has been played in sunshine conditions.\n[B] The follow-up appointment is in 2 weeks.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"27\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_012",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] The soil pH should be between 5.9 and 6.5.\n[B] Revenue from the Asia-Pacific region grew 7%.\n[A] Plant the sunflower seeds 0.25 inches deep.\n[B] The debt-to-equity ratio stands at 2.14.\n[A] Mulch with wood chips to retain moisture.\n[B] Market capitalization reached $476 billion.\n[A] Prune the lettuce back to 13 inches in March.\n[B] Net profit margin improved to 5.5%.\n[A] Expect germination in 9 to 15 days.\n[B] Net profit margin improved to 23.1%.\n[A] Water thoroughly every 4 days during spring.\n[B] Capital expenditure is budgeted at $47 million.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"5.9\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_013",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] The score is 1-3 at the end of the first half.\n[B] Apply the moisturizing cream 2 times per day.\n[A] The score is 4-0 at the end of the first half.\n[B] Exercise for at least 12 minutes daily.\n[A] The corner kick is taken by Ravi.\n[B] Exercise for at least 39 minutes daily.\n[A] The referee issued a red card for the foul.\n[B] Take 500mg of amoxicillin twice daily.\n[A] The match has been played in sunshine conditions.\n[B] Blood pressure reading was 155/69.\n[A] Substitution: Zain replaces Sigrid.\n[B] The follow-up appointment is in 2 weeks.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"1\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_014",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversation B.\n\n[A] Add 45 tablespoons of olive oil to the pan.\n[B] The guided tour starts at 10:00 near the main square.\n[A] Serve on a warm plate alongside rice.\n[B] The train from the airport takes about 39 minutes.\n[A] Stir occasionally until the sauce thickens.\n[B] The guided tour starts at 13:00 near the main square.\n[A] The total cooking time should be about 72 minutes.\n[B] Book a hotel near the central park for the best location.\n[A] Serve on a warm plate alongside bread.\n[B] The rental car pickup is at the east exit.\n[A] Garnish with fresh cilantro before serving.\n[B] The rental car pickup is at the main lobby.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"45\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_medium_015",
  "task_type": "stream_segregation",
  "difficulty": "Medium",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Prune the sunflower back to 18 inches in October.\n[B] The board approved a $57 million share buyback.\n[A] Space each plant at least 7 inches apart.\n[B] Capital expenditure is budgeted at $59 million.\n[A] Plant the lettuce seeds 0.5 inches deep.\n[B] Dividends per share will be $0.59.\n[A] Plant the sunflower seeds 0.25 inches deep.\n[B] The debt-to-equity ratio stands at 1.05.\n[A] Watch for caterpillars \u2014 treat with neem oil if spotted.\n[B] The stock trades at a P/E ratio of 26.3.\n[A] Expect germination in 7 to 15 days.\n[B] The board approved a $274 million share buyback.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\n\nANSWER:\n1. [your answer]",
  "gold_json": "{\"first_number\": \"18\", \"has_breakthrough\": false}"
 },
 {
  "task_id": "stream_hard_016",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about repair instructions).\nCompletely ignore conversation B.\n\n[A] Use a 8mm wrench to loosen the bolt.\n[B] The photographer charges $216 per hour.\n[A] Use a 10mm wrench to loosen the bolt.\n[B] Reserve 29 round tables with 6 chairs each.\n[A] Apply contact cement to both surfaces before joining.\n[B] Parking is available for 88 vehicles.\n[A] Test the operation before restoring power.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** Catering quotes range from $47 to $76 per person.\n[A] Remove the 7 screws from the back panel.\n[B] Catering quotes range from $30 to $60 per person.\n[A] Apply epoxy to both surfaces before joining.\n[B] The cake needs to be ordered 4 days ahead.\n\nQuestion 1: Based ONLY on conversation A (about repair instructions), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"8\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_017",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] Nalini scored from 31 yards out.\n[B] Take 500mg of metformin twice daily.\n[A] Adaeze makes a save from close range.\n[B] Apply the hydrocortisone cream 3 times per day.\n[A] The score is 3-2 at the end of the second half.\n[B] Avoid caffeine for at least 8 days post-procedure.\n[A] The score is 1-2 at the end of the second half.\n[B] Take 200mg of amoxicillin twice daily.\n[A] The match has been played in sunshine conditions.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** Blood pressure reading was 147/75.\n[A] Possession has been 46%-60% so far.\n[B] Limit sodium intake to 1989mg per day.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"31\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_018",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] The score is 0-0 at the end of the third quarter.\n[B] Blood pressure reading was 146/63.\n[A] Possession has been 49%-62% so far.\n[B] Exercise for at least 12 minutes daily.\n[A] The referee issued a red card for the foul.\n[B] Limit sodium intake to 2420mg per day.\n[A] Freya scored from 16 yards out.\n[B] Limit sodium intake to 2278mg per day.\n[A] The score is 1-0 at the end of the third quarter.\n[B] Blood pressure reading was 126/79.\n[A] The corner kick is taken by Bram.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** Apply the antibiotic cream 3 times per day.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"0\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_019",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] Magnus scored from 23 yards out.\n[B] Limit sodium intake to 2171mg per day.\n[A] The match has been played in rain conditions.\n[B] Limit sodium intake to 2388mg per day.\n[A] The referee issued a red card for the foul.\n[B] Schedule a follow-up if symptoms persist beyond 3 days.\n[A] The attendance tonight is 33,528 spectators.\n[B] Take 100mg of amoxicillin twice daily.\n[A] The attendance tonight is 19,696 spectators.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The follow-up appointment is in 8 weeks.\n[A] The match has been played in sunshine conditions.\n[B] Blood pressure reading was 126/91.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"23\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_020",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Harvest when the sunflower reaches 11 inches tall.\n[B] Revenue from the European region grew 25%.\n[A] Add potassium fertilizer once every 2 weeks.\n[B] The quarterly revenue increased by 20% year-over-year.\n[A] Mulch with wood chips to retain moisture.\n[B] Capital expenditure is budgeted at $170 million.\n[A] Prune the sunflower back to 8 inches in October.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** Revenue from the North American region grew 30%.\n[A] Plant the basil seeds 2 inches deep.\n[B] Dividends per share will be $2.72.\n[A] Harvest when the sunflower reaches 14 inches tall.\n[B] Dividends per share will be $2.86.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"11\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_021",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] The soil pH should be between 5.6 and 7.2.\n[B] Dividends per share will be $1.71.\n[A] Expect germination in 5 to 19 days.\n[B] Market capitalization reached $116 billion.\n[A] The soil pH should be between 5.9 and 7.3.\n[B] Revenue from the North American region grew 23%.\n[A] Mulch with straw to retain moisture.\n[B] The quarterly revenue increased by 25% year-over-year.\n[A] Space each plant at least 12 inches apart.\n[B] Dividends per share will be $2.21.\n[A] Plant the lettuce seeds 0.25 inches deep.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The debt-to-equity ratio stands at 1.16.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"5.6\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_022",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about repair instructions).\nCompletely ignore conversation B.\n\n[A] Let the joint set for at least 23 hours.\n[B] Parking is available for 31 vehicles.\n[A] First, disconnect the power supply completely.\n[B] The cake needs to be ordered 9 days ahead.\n[A] Remove the 3 screws from the back panel.\n[B] Send invitations at least 2 weeks in advance.\n[A] Apply epoxy to both surfaces before joining.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The photographer charges $249 per hour.\n[A] Locate the thermal fuse \u2014 it should be near the gate B.\n[B] Send invitations at least 7 weeks in advance.\n[A] Remove the 3 screws from the back panel.\n[B] The venue holds up to 158 guests.\n\nQuestion 1: Based ONLY on conversation A (about repair instructions), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"23\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_hard_023",
  "task_type": "stream_segregation",
  "difficulty": "Hard",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversation B.\n\n[A] The total cooking time should be about 30 minutes.\n[B] The flight departs at 13:30 from terminal 2.\n[A] Remove from heat and let it cool for 24 minutes.\n[B] Book a hotel near the cathedral for the best location.\n[A] Stir occasionally until the sauce thickens.\n[B] Exchange currency at the airport \u2014 the rate is 0.72 to the dollar.\n[A] Dice the zucchini into small cubes.\n[B] Pack an umbrella \u2014 the weather forecast shows rain.\n[A] Let the mixture simmer for 33 minutes.\n[B] Exchange currency at the airport \u2014 the rate is 0.58 to the dollar.\n[A] Add 60 tablespoons of olive oil to the pan.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The rental car pickup is at the east exit.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"30\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_024",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] The score is 2-4 at the end of the third quarter.\n[B] The follow-up appointment is in 3 weeks.\n[A] The corner kick is taken by Bram.\n[B] The recommended daily water intake is 2.7 liters.\n[A] The corner kick is taken by Magnus.\n[B] Schedule a follow-up if symptoms persist beyond 10 days.\n[A] Possession has been 49%-62% so far.\n[B] Apply the antibiotic cream 2 times per day.\n[A] The referee issued a red card for the foul.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The recommended daily water intake is 2.0 liters.\n[A] The match has been played in cold winds conditions.\n[B] The follow-up appointment is in 4 weeks.\n[A] Injury time will be 2 minutes.\n[B] Apply the hydrocortisone cream 3 times per day.\n[A] Injury time will be 5 minutes.\n[B] The test results will be available in 12 business days.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"2\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_025",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Prune the sunflower back to 10 inches in October.\n[B] The board approved a $461 million share buyback.\n[A] Plant the basil seeds 2 inches deep.\n[B] The stock trades at a P/E ratio of 23.1.\n[A] Water thoroughly every 10 days during autumn.\n[B] The quarterly revenue increased by 6% year-over-year.\n[A] Space each plant at least 19 inches apart.\n[B] The board approved a $181 million share buyback.\n[A] Harvest when the lettuce reaches 7 inches tall.\n[B] Revenue from the European region grew 7%.\n[A] Water thoroughly every 11 days during autumn.\n[B] The stock trades at a P/E ratio of 28.7.\n[A] Watch for slugs \u2014 treat with insecticidal soap if spotted.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The stock trades at a P/E ratio of 29.3.\n[A] Plant the sunflower seeds 2 inches deep.\n[B] Market capitalization reached $287 billion.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"10\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_026",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] Injury time will be 5 minutes.\n[B] Apply the antibiotic cream 2 times per day.\n[A] The match has been played in sunshine conditions.\n[B] Blood pressure reading was 130/62.\n[A] The referee issued a yellow card for the foul.\n[B] Schedule a follow-up if symptoms persist beyond 4 days.\n[A] The attendance tonight is 70,976 spectators.\n[B] Schedule a follow-up if symptoms persist beyond 10 days.\n[A] Possession has been 61%-51% so far.\n[B] The test results will be available in 11 business days.\n[A] The corner kick is taken by Zora.\n[B] Limit sodium intake to 1656mg per day.\n[A] Injury time will be 4 minutes.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** Take 100mg of ibuprofen twice daily.\n[A] The corner kick is taken by Greta.\n[B] Apply the hydrocortisone cream 3 times per day.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"5\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_027",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Prune the basil back to 14 inches in October.\n[B] Capital expenditure is budgeted at $88 million.\n[A] Plant the tomato seeds 0.25 inches deep.\n[B] Dividends per share will be $4.46.\n[A] Expect germination in 7 to 20 days.\n[B] Capital expenditure is budgeted at $24 million.\n[A] Plant the basil seeds 1 inches deep.\n[B] Capital expenditure is budgeted at $174 million.\n[A] Water thoroughly every 12 days during spring.\n[B] Revenue from the North American region grew 30%.\n[A] The soil pH should be between 6.2 and 7.4.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The stock trades at a P/E ratio of 26.3.\n[A] The soil pH should be between 5.5 and 7.3.\n[B] Capital expenditure is budgeted at $154 million.\n[A] Harvest when the lettuce reaches 26 inches tall.\n[B] Dividends per share will be $0.72.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"14\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_028",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about sports commentary).\nCompletely ignore conversation B.\n\n[A] Freya scored from 18 yards out.\n[B] Avoid caffeine for at least 8 days post-procedure.\n[A] The corner kick is taken by Kenji.\n[B] Take 100mg of ibuprofen twice daily.\n[A] Magnus makes a save from close range.\n[B] Take 250mg of metformin twice daily.\n[A] Possession has been 44%-40% so far.\n[B] The test results will be available in 4 business days.\n[A] The corner kick is taken by Dmitri.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The recommended daily water intake is 1.8 liters.\n[A] The referee issued a yellow card for the foul.\n[B] Limit sodium intake to 1752mg per day.\n[A] The corner kick is taken by Femi.\n[B] Avoid gluten for at least 6 days post-procedure.\n[A] The score is 2-3 at the end of the first half.\n[B] Schedule a follow-up if symptoms persist beyond 4 days.\n\nQuestion 1: Based ONLY on conversation A (about sports commentary), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"18\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_029",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversation B.\n\n[A] Remove from heat and let it cool for 22 minutes.\n[B] The flight departs at 8:00 from terminal 1.\n[A] Let the mixture simmer for 21 minutes.\n[B] Check out is at 10:00 \u2014 leave bags at reception.\n[A] Serve on a warm plate alongside rice.\n[B] Budget approximately $36 per day for meals.\n[A] The total cooking time should be about 39 minutes.\n[B] Pack sunscreen \u2014 the weather forecast shows cold winds.\n[A] Serve on a warm plate alongside salad.\n[B] Budget approximately $95 per day for meals.\n[A] First, preheat the oven to 265 degrees.\n[B] The museum on Zora Street is open until 20:00.\n[A] Remove from heat and let it cool for 22 minutes.\n[B] Book a hotel near the central park for the best location.\n[A] First, preheat the oven to 448 degrees.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** Exchange currency at the airport \u2014 the rate is 1.13 to the dollar.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"22\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_030",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversation B.\n\n[A] Add 271 tablespoons of olive oil to the pan.\n[B] Book a hotel near the central park for the best location.\n[A] Stir occasionally until the sauce thickens.\n[B] The flight departs at 19:30 from terminal 1.\n[A] First, preheat the oven to 150 degrees.\n[B] The guided tour starts at 14:00 near the main square.\n[A] Remove from heat and let it cool for 33 minutes.\n[B] Budget approximately $60 per day for meals.\n[A] Dice the pepper into small cubes.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** Book a hotel near the central park for the best location.\n[A] Remove from heat and let it cool for 18 minutes.\n[B] The guided tour starts at 9:00 near the main square.\n[A] Remove from heat and let it cool for 13 minutes.\n[B] The rental car pickup is at gate B.\n[A] The total cooking time should be about 54 minutes.\n[B] The guided tour starts at 13:00 near the main square.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"271\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_expert_031",
  "task_type": "stream_segregation",
  "difficulty": "Expert",
  "prompt": "Below are two interleaved conversations marked [A] and [B].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversation B.\n\n[A] Water thoroughly every 3 days during summer.\n[B] The stock trades at a P/E ratio of 18.5.\n[A] Space each plant at least 12 inches apart.\n[B] Operating costs are projected at $172 million.\n[A] Water thoroughly every 14 days during autumn.\n[B] The board approved a $483 million share buyback.\n[A] Water thoroughly every 3 days during summer.\n[B] The quarterly revenue increased by 20% year-over-year.\n[A] Plant the basil seeds 1 inches deep.\n[B] Market capitalization reached $21 billion.\n[A] Water thoroughly every 11 days during spring.\n[B] The debt-to-equity ratio stands at 1.22.\n[A] The soil pH should be between 6.0 and 7.5.\n[B] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** Dividends per share will be $2.55.\n[A] Plant the sunflower seeds 2 inches deep.\n[B] Revenue from the North American region grew 24%.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation B? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"3\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_032",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversations B and C.\n\n[A] First, preheat the oven to 392 degrees.\n[B] Pack a warm jacket \u2014 the weather forecast shows rain.\n[C] Dividends per share will be $3.65.\n[A] Season with salt, pepper, and a pinch of turmeric.\n[B] Book a hotel near the old market for the best location.\n[C] Revenue from the Asia-Pacific region grew 21%.\n[A] Season with salt, pepper, and a pinch of paprika.\n[B] The flight departs at 14:15 from terminal 3.\n[C] Capital expenditure is budgeted at $162 million.\n[A] Add 421 tablespoons of olive oil to the pan.\n[B] Check out is at 12:00 \u2014 leave bags at reception.\n[C] Operating costs are projected at $54 million.\n[A] Dice the zucchini into small cubes.\n[B] Pack an umbrella \u2014 the weather forecast shows cold winds.\n[C] Capital expenditure is budgeted at $97 million.\n[A] Dice the celery into small cubes.\n[B] The rental car pickup is at gate B.\n[C] Net profit margin improved to 15.3%.\n[A] First, preheat the oven to 240 degrees.\n[B] The rental car pickup is at the main lobby.\n[C] Net profit margin improved to 19.2%.\n[A] Dice the carrot into small cubes.\n[B] Budget approximately $60 per day for meals.\n[C] Market capitalization reached $280 billion.\n[A] Season with salt, pepper, and a pinch of cumin.\n[B] The museum on Ravi Street is open until 18:00.\n[C] The debt-to-equity ratio stands at 1.16.\n[A] Garnish with fresh basil before serving.\n[B] The flight departs at 17:45 from terminal 1.\n[C] The debt-to-equity ratio stands at 0.47.\n[A] Stir occasionally until the sauce thickens.\n[B] Book a hotel near the central park for the best location.\n[C] The board approved a $239 million share buyback.\n[A] Remove from heat and let it cool for 22 minutes.\n[B] Pack sunscreen \u2014 the weather forecast shows sunshine.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** Capital expenditure is budgeted at $38 million.\n[A] First, preheat the oven to 378 degrees.\n[B] Book a hotel near the cathedral for the best location.\n[C] Net profit margin improved to 15.4%.\n[A] Dice the pepper into small cubes.\n[B] The train from the airport takes about 38 minutes.\n[C] The stock trades at a P/E ratio of 11.5.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"392\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_033",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversations B and C.\n\n[A] First, preheat the oven to 447 degrees.\n[B] The rental car pickup is at the east exit.\n[C] Revenue from the North American region grew 21%.\n[A] Add 122 tablespoons of olive oil to the pan.\n[B] Exchange currency at the airport \u2014 the rate is 0.87 to the dollar.\n[C] Market capitalization reached $404 billion.\n[A] The total cooking time should be about 81 minutes.\n[B] The guided tour starts at 10:00 near the main square.\n[C] Capital expenditure is budgeted at $51 million.\n[A] Garnish with fresh parsley before serving.\n[B] The flight departs at 9:00 from terminal 2.\n[C] Market capitalization reached $483 billion.\n[A] Let the mixture simmer for 26 minutes.\n[B] The flight departs at 13:45 from terminal 2.\n[C] The quarterly revenue increased by 14% year-over-year.\n[A] The total cooking time should be about 30 minutes.\n[B] The flight departs at 10:00 from terminal 4.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** Revenue from the European region grew 12%.\n[A] Let the mixture simmer for 28 minutes.\n[B] Book a hotel near the old market for the best location.\n[C] Dividends per share will be $4.28.\n[A] Let the mixture simmer for 16 minutes.\n[B] The flight departs at 12:00 from terminal 3.\n[C] The quarterly revenue increased by 10% year-over-year.\n[A] Add 433 tablespoons of olive oil to the pan.\n[B] Budget approximately $110 per day for meals.\n[C] Capital expenditure is budgeted at $123 million.\n[A] Stir occasionally until the sauce thickens.\n[B] Exchange currency at the airport \u2014 the rate is 1.27 to the dollar.\n[C] Net profit margin improved to 24.6%.\n[A] Add 326 tablespoons of olive oil to the pan.\n[B] Check out is at 11:00 \u2014 leave bags at reception.\n[C] The stock trades at a P/E ratio of 15.3.\n[A] Remove from heat and let it cool for 42 minutes.\n[B] Exchange currency at the airport \u2014 the rate is 0.57 to the dollar.\n[C] Revenue from the Asia-Pacific region grew 13%.\n[A] First, preheat the oven to 357 degrees.\n[B] The guided tour starts at 11:00 near the main square.\n[C] Revenue from the European region grew 3%.\n[A] Dice the zucchini into small cubes.\n[B] The rental car pickup is at the east exit.\n[C] Market capitalization reached $72 billion.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"447\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_034",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversations B and C.\n\n[A] Prune the sunflower back to 7 inches in October.\n[B] The corner kick is taken by Ravi.\n[C] The rental car pickup is at the main lobby.\n[A] Space each plant at least 9 inches apart.\n[B] The referee issued a red card for the foul.\n[C] Budget approximately $81 per day for meals.\n[A] Harvest when the tomato reaches 24 inches tall.\n[B] The referee issued a red card for the foul.\n[C] The rental car pickup is at the main lobby.\n[A] Mulch with leaf compost to retain moisture.\n[B] The score is 1-4 at the end of the third quarter.\n[C] The flight departs at 9:00 from terminal 1.\n[A] Add potassium fertilizer once every 3 weeks.\n[B] The corner kick is taken by Zain.\n[C] The guided tour starts at 14:00 near the main square.\n[A] Water thoroughly every 10 days during spring.\n[B] Injury time will be 5 minutes.\n[C] Pack an umbrella \u2014 the weather forecast shows cold winds.\n[A] The soil pH should be between 6.3 and 7.1.\n[B] Substitution: Idris replaces Joelle.\n[C] Pack sunscreen \u2014 the weather forecast shows cold winds.\n[A] Prune the lettuce back to 12 inches in April.\n[B] Injury time will be 3 minutes.\n[C] Check out is at 12:00 \u2014 leave bags at reception.\n[A] Prune the lettuce back to 17 inches in October.\n[B] Substitution: Nico replaces Kenji.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The flight departs at 9:30 from terminal 2.\n[A] Add potassium fertilizer once every 5 weeks.\n[B] The corner kick is taken by Orla.\n[C] Book a hotel near the central park for the best location.\n[A] Mulch with wood chips to retain moisture.\n[B] The referee issued a red card for the foul.\n[C] Check out is at 12:00 \u2014 leave bags at reception.\n[A] The soil pH should be between 6.2 and 7.4.\n[B] Zain scored from 32 yards out.\n[C] Budget approximately $101 per day for meals.\n[A] Space each plant at least 18 inches apart.\n[B] Sigrid makes a save from close range.\n[C] Budget approximately $55 per day for meals.\n[A] Add potassium fertilizer once every 2 weeks.\n[B] The referee issued a yellow card for the foul.\n[C] Budget approximately $139 per day for meals.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"7\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_035",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversations B and C.\n\n[A] Add nitrogen-rich fertilizer once every 2 weeks.\n[B] The score is 2-0 at the end of the third quarter.\n[C] The guided tour starts at 11:00 near the main square.\n[A] Water thoroughly every 2 days during spring.\n[B] The match has been played in sunshine conditions.\n[C] Book a hotel near the cathedral for the best location.\n[A] Add potassium fertilizer once every 7 weeks.\n[B] The match has been played in cold winds conditions.\n[C] Budget approximately $149 per day for meals.\n[A] Plant the basil seeds 2 inches deep.\n[B] The match has been played in cold winds conditions.\n[C] Book a hotel near the central park for the best location.\n[A] Watch for slugs \u2014 treat with neem oil if spotted.\n[B] Possession has been 61%-51% so far.\n[C] Exchange currency at the airport \u2014 the rate is 0.97 to the dollar.\n[A] Harvest when the tomato reaches 12 inches tall.\n[B] Possession has been 43%-44% so far.\n[C] The flight departs at 13:15 from terminal 3.\n[A] Space each plant at least 22 inches apart.\n[B] Colette scored from 10 yards out.\n[C] Book a hotel near the central park for the best location.\n[A] The soil pH should be between 6.3 and 7.1.\n[B] The match has been played in sunshine conditions.\n[C] The flight departs at 12:45 from terminal 3.\n[A] Space each plant at least 6 inches apart.\n[B] Olena makes a save from close range.\n[C] The museum on Viktor Street is open until 19:00.\n[A] Water thoroughly every 8 days during autumn.\n[B] The score is 0-4 at the end of the second half.\n[C] The guided tour starts at 12:00 near the main square.\n[A] The soil pH should be between 5.7 and 6.9.\n[B] The referee issued a yellow card for the foul.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The rental car pickup is at gate B.\n[A] Harvest when the sunflower reaches 22 inches tall.\n[B] Qadir makes a save from close range.\n[C] The museum on Femi Street is open until 17:00.\n[A] Mulch with straw to retain moisture.\n[B] Possession has been 43%-46% so far.\n[C] The train from the airport takes about 6 minutes.\n[A] Harvest when the tomato reaches 33 inches tall.\n[B] The attendance tonight is 28,263 spectators.\n[C] Pack an umbrella \u2014 the weather forecast shows rain.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"2\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_036",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversations B and C.\n\n[A] Expect germination in 10 to 12 days.\n[B] The referee issued a yellow card for the foul.\n[C] The guided tour starts at 10:00 near the main square.\n[A] Expect germination in 7 to 12 days.\n[B] The attendance tonight is 31,466 spectators.\n[C] The museum on Lumi Street is open until 20:00.\n[A] The soil pH should be between 6.1 and 7.4.\n[B] Injury time will be 2 minutes.\n[C] Check out is at 12:00 \u2014 leave bags at reception.\n[A] Expect germination in 10 to 16 days.\n[B] The attendance tonight is 22,742 spectators.\n[C] Budget approximately $127 per day for meals.\n[A] Watch for slugs \u2014 treat with diatomaceous earth if spotted.\n[B] Yuki scored from 9 yards out.\n[C] The train from the airport takes about 16 minutes.\n[A] Plant the lettuce seeds 1 inches deep.\n[B] Femi makes a save from close range.\n[C] The train from the airport takes about 39 minutes.\n[A] Plant the tomato seeds 1 inches deep.\n[B] The corner kick is taken by Hana.\n[C] The rental car pickup is at gate B.\n[A] Prune the tomato back to 10 inches in March.\n[B] The score is 4-4 at the end of the first half.\n[C] Book a hotel near the cathedral for the best location.\n[A] Prune the lettuce back to 17 inches in April.\n[B] The corner kick is taken by Nalini.\n[C] The guided tour starts at 10:00 near the main square.\n[A] The soil pH should be between 5.7 and 7.4.\n[B] The attendance tonight is 25,945 spectators.\n[C] The museum on Kenji Street is open until 20:00.\n[A] Plant the basil seeds 2 inches deep.\n[B] Injury time will be 5 minutes.\n[C] Book a hotel near the cathedral for the best location.\n[A] Watch for caterpillars \u2014 treat with insecticidal soap if spotted.\n[B] The match has been played in rain conditions.\n[C] The flight departs at 8:00 from terminal 1.\n[A] Mulch with straw to retain moisture.\n[B] The match has been played in rain conditions.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The train from the airport takes about 37 minutes.\n[A] Harvest when the tomato reaches 29 inches tall.\n[B] The score is 2-4 at the end of the second half.\n[C] The train from the airport takes about 5 minutes.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"10\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_037",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversations B and C.\n\n[A] Space each plant at least 19 inches apart.\n[B] The corner kick is taken by Joelle.\n[C] The museum on Freya Street is open until 19:00.\n[A] Space each plant at least 12 inches apart.\n[B] Injury time will be 4 minutes.\n[C] The rental car pickup is at the east exit.\n[A] Expect germination in 7 to 18 days.\n[B] The attendance tonight is 72,168 spectators.\n[C] The museum on Dariush Street is open until 21:00.\n[A] Water thoroughly every 8 days during spring.\n[B] Nalini scored from 18 yards out.\n[C] Check out is at 12:00 \u2014 leave bags at reception.\n[A] Prune the lettuce back to 15 inches in March.\n[B] The match has been played in cold winds conditions.\n[C] The train from the airport takes about 42 minutes.\n[A] Plant the basil seeds 0.5 inches deep.\n[B] The score is 4-4 at the end of the second half.\n[C] The rental car pickup is at the main lobby.\n[A] Prune the lettuce back to 6 inches in October.\n[B] The match has been played in cold winds conditions.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The guided tour starts at 13:00 near the main square.\n[A] Space each plant at least 7 inches apart.\n[B] Injury time will be 2 minutes.\n[C] The rental car pickup is at the main lobby.\n[A] Expect germination in 6 to 21 days.\n[B] Paloma scored from 35 yards out.\n[C] Exchange currency at the airport \u2014 the rate is 1.41 to the dollar.\n[A] Water thoroughly every 2 days during autumn.\n[B] The attendance tonight is 46,816 spectators.\n[C] The flight departs at 14:00 from terminal 4.\n[A] Plant the tomato seeds 1 inches deep.\n[B] The attendance tonight is 29,416 spectators.\n[C] The museum on Haruto Street is open until 20:00.\n[A] Expect germination in 7 to 13 days.\n[B] The match has been played in sunshine conditions.\n[C] The guided tour starts at 9:00 near the main square.\n[A] Plant the lettuce seeds 2 inches deep.\n[B] Possession has been 53%-57% so far.\n[C] Exchange currency at the airport \u2014 the rate is 0.65 to the dollar.\n[A] Watch for slugs \u2014 treat with insecticidal soap if spotted.\n[B] Injury time will be 4 minutes.\n[C] Pack a warm jacket \u2014 the weather forecast shows cold winds.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"19\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_038",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about cooking recipe).\nCompletely ignore conversations B and C.\n\n[A] Remove from heat and let it cool for 9 minutes.\n[B] If the issue persists, check the wiring harness.\n[C] The test results will be available in 11 business days.\n[A] The total cooking time should be about 33 minutes.\n[B] Test the operation before restoring power.\n[C] The recommended daily water intake is 1.9 liters.\n[A] Let the mixture simmer for 38 minutes.\n[B] Remove the 2 screws from the back panel.\n[C] The recommended daily water intake is 2.5 liters.\n[A] First, preheat the oven to 305 degrees.\n[B] Replace the worn gasket with the new one from the kit.\n[C] Blood pressure reading was 145/66.\n[A] Add 441 tablespoons of olive oil to the pan.\n[B] Let the joint set for at least 4 hours.\n[C] The follow-up appointment is in 7 weeks.\n[A] Let the mixture simmer for 40 minutes.\n[B] First, disconnect the power supply completely.\n[C] Take 250mg of metformin twice daily.\n[A] Serve on a warm plate alongside potatoes.\n[B] Locate the relay switch \u2014 it should be near the gate B.\n[C] Exercise for at least 30 minutes daily.\n[A] Dice the celery into small cubes.\n[B] Remove the 2 screws from the back panel.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The recommended daily water intake is 1.9 liters.\n[A] Let the mixture simmer for 40 minutes.\n[B] Use a 8mm wrench to loosen the bolt.\n[C] Avoid gluten for at least 8 days post-procedure.\n[A] Season with salt, pepper, and a pinch of cumin.\n[B] Locate the thermal fuse \u2014 it should be near the gate B.\n[C] Avoid alcohol for at least 4 days post-procedure.\n[A] Season with salt, pepper, and a pinch of cumin.\n[B] Reattach the panel and tighten screws to 24 Nm.\n[C] The follow-up appointment is in 2 weeks.\n[A] Stir occasionally until the sauce thickens.\n[B] Reattach the panel and tighten screws to 14 Nm.\n[C] The test results will be available in 5 business days.\n[A] Dice the zucchini into small cubes.\n[B] Locate the thermal fuse \u2014 it should be near the the main lobby.\n[C] The recommended daily water intake is 1.7 liters.\n[A] Season with salt, pepper, and a pinch of paprika.\n[B] Reattach the panel and tighten screws to 24 Nm.\n[C] The follow-up appointment is in 6 weeks.\n\nQuestion 1: Based ONLY on conversation A (about cooking recipe), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"9\", \"has_breakthrough\": true}"
 },
 {
  "task_id": "stream_frontier_039",
  "task_type": "stream_segregation",
  "difficulty": "Frontier",
  "prompt": "Below are THREE interleaved conversations marked [A], [B], and [C].\nAnswer questions ONLY about conversation A (about gardening guide).\nCompletely ignore conversations B and C.\n\n[A] Plant the lettuce seeds 2 inches deep.\n[B] Injury time will be 2 minutes.\n[C] Check out is at 12:00 \u2014 leave bags at reception.\n[A] Harvest when the lettuce reaches 35 inches tall.\n[B] Possession has been 36%-50% so far.\n[C] Book a hotel near the old market for the best location.\n[A] Prune the basil back to 13 inches in October.\n[B] The match has been played in rain conditions.\n[C] Budget approximately $88 per day for meals.\n[A] Water thoroughly every 13 days during summer.\n[B] Injury time will be 2 minutes.\n[C] The museum on Xander Street is open until 21:00.\n[A] Watch for aphids \u2014 treat with insecticidal soap if spotted.\n[B] Sigrid makes a save from close range.\n[C] The museum on Orla Street is open until 21:00.\n[A] Space each plant at least 20 inches apart.\n[B] Substitution: Viktor replaces Viktor.\n[C] The museum on Kenji Street is open until 18:00.\n[A] Plant the basil seeds 0.5 inches deep.\n[B] The score is 1-3 at the end of the third quarter.\n[C] ***** ALERT ALERT ALERT: URGENT PRIORITY NOTICE \u2014 THIS IS AN ALERT \u2014 immediate attention required. ALERT ***** The flight departs at 20:30 from terminal 5.\n[A] Prune the lettuce back to 11 inches in October.\n[B] The match has been played in cold winds conditions.\n[C] The train from the airport takes about 10 minutes.\n[A] Add nitrogen-rich fertilizer once every 8 weeks.\n[B] Injury time will be 2 minutes.\n[C] Budget approximately $127 per day for meals.\n[A] Water thoroughly every 6 days during autumn.\n[B] Injury time will be 2 minutes.\n[C] Budget approximately $102 per day for meals.\n[A] Water thoroughly every 9 days during summer.\n[B] The score is 2-3 at the end of the third quarter.\n[C] The guided tour starts at 14:00 near the main square.\n[A] Expect germination in 7 to 12 days.\n[B] The score is 2-2 at the end of the second half.\n[C] Pack a warm jacket \u2014 the weather forecast shows rain.\n[A] Harvest when the tomato reaches 6 inches tall.\n[B] Injury time will be 4 minutes.\n[C] The train from the airport takes about 32 minutes.\n[A] Space each plant at least 24 inches apart.\n[B] The corner kick is taken by Tariq.\n[C] Pack a warm jacket \u2014 the weather forecast shows cold winds.\n\nQuestion 1: Based ONLY on conversation A (about gardening guide), what is the first Arabic numeral (e.g. 3, 450, 12.5) that appears?\nQuestion 2: Was the word \"ALERT\" mentioned anywhere in conversation C? (yes/no)\n\nANSWER:\n1. [your answer]\n2. [yes/no]",
  "gold_json": "{\"first_number\": \"2\", \"has_breakthrough\": true}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['sustained', 'stream_segregation']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "sustained": cogattention_sustained,
    "stream_segregation": cogattention_stream_segregation,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Sustained Attention")
